# Customer Churn Prediction - Exploratory Data Analysis

## Project Overview
This project analyzes customer churn patterns in a telecommunications company. We aim to:
- Understand customer behavior and churn drivers
- Identify high-risk customer segments
- Build predictive models to enable proactive retention strategies

## Dataset
- **Size**: 10,000 customer records
- **Features**: 23 attributes including demographics, services, billing, and engagement metrics
- **Target**: Binary churn indicator (1 = churned, 0 = retained)

## 1. Setup and Data Generation

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Set random seed
np.random.seed(42)

In [ ]:
# Generate synthetic customer data
print("Generating customer churn dataset...")

n_customers = 10000

# Customer IDs
customer_ids = [f'CUST{str(i).zfill(6)}' for i in range(1, n_customers + 1)]

# Demographics
genders = np.random.choice(['Male', 'Female'], n_customers, p=[0.52, 0.48])
ages = np.random.normal(42, 15, n_customers).clip(18, 80).astype(int)
senior_citizen = (ages >= 65).astype(int)

# Geographic data
states = np.random.choice(['CA', 'TX', 'FL', 'NY', 'PA', 'IL', 'OH', 'GA', 'NC', 'MI'], n_customers)

# Account information
tenure_months = np.random.exponential(24, n_customers).clip(0, 72).astype(int)
contract_types = np.random.choice(['Month-to-Month', 'One Year', 'Two Year'], n_customers, 
                                   p=[0.55, 0.25, 0.20])

# Service features
phone_service = np.random.choice([0, 1], n_customers, p=[0.1, 0.9])
multiple_lines = np.where(phone_service == 1, 
                          np.random.choice([0, 1], n_customers, p=[0.5, 0.5]), 
                          0)

internet_service = np.random.choice(['No', 'DSL', 'Fiber Optic'], n_customers, 
                                     p=[0.2, 0.35, 0.45])
online_security = np.where(internet_service != 'No',
                           np.random.choice([0, 1], n_customers, p=[0.6, 0.4]),
                           0)
online_backup = np.where(internet_service != 'No',
                         np.random.choice([0, 1], n_customers, p=[0.55, 0.45]),
                         0)
device_protection = np.where(internet_service != 'No',
                             np.random.choice([0, 1], n_customers, p=[0.6, 0.4]),
                             0)
tech_support = np.where(internet_service != 'No',
                        np.random.choice([0, 1], n_customers, p=[0.65, 0.35]),
                        0)
streaming_tv = np.where(internet_service != 'No',
                        np.random.choice([0, 1], n_customers, p=[0.5, 0.5]),
                        0)
streaming_movies = np.where(internet_service != 'No',
                            np.random.choice([0, 1], n_customers, p=[0.5, 0.5]),
                            0)

# Billing information
paperless_billing = np.random.choice([0, 1], n_customers, p=[0.4, 0.6])
payment_methods = np.random.choice(['Electronic Check', 'Mailed Check', 'Bank Transfer', 'Credit Card'],
                                    n_customers, p=[0.33, 0.15, 0.22, 0.30])

# Calculate monthly charges based on services
base_charge = 20
monthly_charges = base_charge + \
                  (phone_service * 10) + \
                  (multiple_lines * 5) + \
                  (np.where(internet_service == 'DSL', 30, 0)) + \
                  (np.where(internet_service == 'Fiber Optic', 50, 0)) + \
                  (online_security * 5) + \
                  (online_backup * 5) + \
                  (device_protection * 5) + \
                  (tech_support * 5) + \
                  (streaming_tv * 10) + \
                  (streaming_movies * 10)

monthly_charges = monthly_charges + np.random.normal(0, 3, n_customers)
monthly_charges = monthly_charges.clip(20, 120).round(2)

# Total charges
total_charges = (monthly_charges * tenure_months).round(2)
total_charges = np.where(tenure_months == 0, 0, total_charges)

# Calculate churn with realistic patterns
churn_score = np.zeros(n_customers)
churn_score += (contract_types == 'Month-to-Month') * 0.30
churn_score += (tenure_months < 6) * 0.25
churn_score += (monthly_charges > 70) * 0.15
churn_score += (online_security == 0) * 0.10
churn_score += (tech_support == 0) * 0.10
churn_score += (payment_methods == 'Electronic Check') * 0.15
churn_score += (internet_service == 'Fiber Optic') * 0.10

churn_score -= (contract_types == 'Two Year') * 0.25
churn_score -= (tenure_months > 24) * 0.20
churn_score -= (paperless_billing == 1) * 0.05
churn_score -= (payment_methods == 'Bank Transfer') * 0.05

churn_score += np.random.normal(0, 0.1, n_customers)
churn_prob_final = 1 / (1 + np.exp(-churn_score))
churn = (np.random.random(n_customers) < churn_prob_final).astype(int)

# Additional metrics
satisfaction_score = (100 - churn_prob_final * 100 + np.random.normal(0, 10, n_customers)).clip(0, 100).round(1)
support_tickets = np.where(churn == 1,
                           np.random.poisson(3, n_customers),
                           np.random.poisson(1, n_customers))

# Create DataFrame
df = pd.DataFrame({
    'CustomerID': customer_ids,
    'Gender': genders,
    'Age': ages,
    'SeniorCitizen': senior_citizen,
    'State': states,
    'TenureMonths': tenure_months,
    'ContractType': contract_types,
    'PhoneService': phone_service,
    'MultipleLines': multiple_lines,
    'InternetService': internet_service,
    'OnlineSecurity': online_security,
    'OnlineBackup': online_backup,
    'DeviceProtection': device_protection,
    'TechSupport': tech_support,
    'StreamingTV': streaming_tv,
    'StreamingMovies': streaming_movies,
    'PaperlessBilling': paperless_billing,
    'PaymentMethod': payment_methods,
    'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges,
    'SatisfactionScore': satisfaction_score,
    'SupportTickets': support_tickets,
    'Churn': churn
})

# Save to CSV
df.to_csv('../raw/customer_churn_data.csv', index=False)

print(f"✓ Dataset generated successfully!")
print(f"  Total records: {len(df):,}")
print(f"  Churn rate: {df['Churn'].mean():.2%}")

## 2. Initial Data Exploration

In [ ]:
# Display first few rows
print("Dataset Preview:")
df.head(10)

In [ ]:
# Dataset structure
print("Dataset Information:")
print("=" * 60)
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:")
print("=" * 60)
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print("=" * 60)
missing = df.isnull().sum()
if missing.sum() == 0:
    print("✓ No missing values found!")
else:
    print(missing[missing > 0])

## 3. Target Variable Analysis

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
churn_counts = df['Churn'].value_counts()
axes[0].bar(['Retained', 'Churned'], churn_counts.values, color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[0].set_ylabel('Number of Customers', fontsize=12, fontweight='bold')
axes[0].set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 100, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom', fontweight='bold')

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'], autopct='%1.1f%%', 
            colors=colors, startangle=90, explode=[0, 0.05], shadow=True, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Churn Rate', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/01_churn_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nChurn Statistics:")
print(f"  Retained: {churn_counts[0]:,} ({churn_counts[0]/len(df)*100:.2f}%)")
print(f"  Churned:  {churn_counts[1]:,} ({churn_counts[1]/len(df)*100:.2f}%)")

## 4. Demographic Analysis

In [ ]:
# Age and gender analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Age distribution by churn
for churn_val, label, color in [(0, 'Retained', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
    axes[0, 0].hist(df[df['Churn']==churn_val]['Age'], bins=30, alpha=0.6, label=label, color=color, edgecolor='black')
axes[0, 0].set_xlabel('Age', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Age Distribution by Churn Status', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Gender vs churn
gender_churn = pd.crosstab(df['Gender'], df['Churn'], normalize='index') * 100
gender_churn.plot(kind='bar', ax=axes[0, 1], color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[0, 1].set_xlabel('Gender', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Percentage', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Churn Rate by Gender', fontsize=12, fontweight='bold')
axes[0, 1].legend(['Retained', 'Churned'])
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)
axes[0, 1].grid(axis='y', alpha=0.3)

# Senior citizen vs churn
senior_churn = pd.crosstab(df['SeniorCitizen'], df['Churn'], normalize='index') * 100
senior_churn.plot(kind='bar', ax=axes[1, 0], color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[1, 0].set_xlabel('Senior Citizen', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Percentage', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Churn Rate by Senior Citizen Status', fontsize=12, fontweight='bold')
axes[1, 0].legend(['Retained', 'Churned'])
axes[1, 0].set_xticklabels(['Non-Senior', 'Senior'], rotation=0)
axes[1, 0].grid(axis='y', alpha=0.3)

# Age boxplot by churn
df.boxplot(column='Age', by='Churn', ax=axes[1, 1], patch_artist=True)
axes[1, 1].set_xlabel('Churn Status', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Age', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Age Distribution by Churn', fontsize=12, fontweight='bold')
axes[1, 1].set_xticklabels(['Retained', 'Churned'])
plt.suptitle('')

plt.tight_layout()
plt.savefig('../visualizations/02_demographics_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Contract and Tenure Analysis

In [ ]:
# Contract type and tenure analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Contract type vs churn
contract_churn = pd.crosstab(df['ContractType'], df['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', ax=axes[0, 0], color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[0, 0].set_xlabel('Contract Type', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Percentage', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Churn Rate by Contract Type', fontsize=12, fontweight='bold')
axes[0, 0].legend(['Retained', 'Churned'])
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45, ha='right')
axes[0, 0].grid(axis='y', alpha=0.3)

# Tenure distribution by churn
for churn_val, label, color in [(0, 'Retained', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
    axes[0, 1].hist(df[df['Churn']==churn_val]['TenureMonths'], bins=30, alpha=0.6, label=label, color=color, edgecolor='black')
axes[0, 1].set_xlabel('Tenure (Months)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Tenure Distribution by Churn Status', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Tenure groups
df['TenureGroup'] = pd.cut(df['TenureMonths'], bins=[0, 12, 24, 48, 72], 
                            labels=['0-12 months', '13-24 months', '25-48 months', '49-72 months'])
tenure_churn = pd.crosstab(df['TenureGroup'], df['Churn'], normalize='index') * 100
tenure_churn.plot(kind='bar', ax=axes[1, 0], color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[1, 0].set_xlabel('Tenure Group', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Churn Rate (%)', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Churn Rate by Tenure Group', fontsize=12, fontweight='bold')
axes[1, 0].legend(['Retained', 'Churned'])
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45, ha='right')
axes[1, 0].grid(axis='y', alpha=0.3)

# Payment method vs churn
payment_churn = pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index') * 100
payment_churn.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[1, 1].set_xlabel('Payment Method', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Churn Rate (%)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Churn Rate by Payment Method', fontsize=12, fontweight='bold')
axes[1, 1].legend(['Retained', 'Churned'])
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45, ha='right')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/03_contract_tenure_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Service Analysis

In [ ]:
# Internet service and add-ons analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Internet service vs churn
internet_churn = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100
internet_churn.plot(kind='bar', ax=axes[0, 0], color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[0, 0].set_xlabel('Internet Service', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Churn Rate (%)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Churn Rate by Internet Service Type', fontsize=12, fontweight='bold')
axes[0, 0].legend(['Retained', 'Churned'])
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=0)
axes[0, 0].grid(axis='y', alpha=0.3)

# Service add-ons impact
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
service_impact = []
for service in services:
    churn_rate_with = df[df[service]==1]['Churn'].mean() * 100
    churn_rate_without = df[df[service]==0]['Churn'].mean() * 100
    service_impact.append([churn_rate_with, churn_rate_without])

x = np.arange(len(services))
width = 0.35
axes[0, 1].bar(x - width/2, [s[0] for s in service_impact], width, label='With Service', color='#3498db', alpha=0.8, edgecolor='black')
axes[0, 1].bar(x + width/2, [s[1] for s in service_impact], width, label='Without Service', color='#e67e22', alpha=0.8, edgecolor='black')
axes[0, 1].set_xlabel('Service Type', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Churn Rate (%)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Impact of Services on Churn Rate', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(['Security', 'Backup', 'Protection', 'Support'], rotation=45, ha='right')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Streaming services
streaming_data = [
    df[df['StreamingTV']==1]['Churn'].mean() * 100,
    df[df['StreamingMovies']==1]['Churn'].mean() * 100,
    df[(df['StreamingTV']==1) & (df['StreamingMovies']==1)]['Churn'].mean() * 100,
    df[(df['StreamingTV']==0) & (df['StreamingMovies']==0)]['Churn'].mean() * 100
]
categories = ['TV Only', 'Movies Only', 'Both', 'Neither']
axes[1, 0].barh(categories, streaming_data, color=['#9b59b6', '#1abc9c', '#f39c12', '#95a5a6'], alpha=0.8, edgecolor='black')
axes[1, 0].set_xlabel('Churn Rate (%)', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Churn Rate by Streaming Services', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# Service count impact
df['ServiceCount'] = (df['OnlineSecurity'] + df['OnlineBackup'] + 
                      df['DeviceProtection'] + df['TechSupport'] + 
                      df['StreamingTV'] + df['StreamingMovies'])
service_count_churn = df.groupby('ServiceCount')['Churn'].agg(['mean', 'count'])
service_count_churn['mean'] *= 100

axes[1, 1].bar(service_count_churn.index, service_count_churn['mean'], 
               color='#e74c3c', alpha=0.8, edgecolor='black')
axes[1, 1].set_xlabel('Number of Additional Services', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Churn Rate (%)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Churn Rate by Service Count', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/04_service_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Financial Analysis

In [ ]:
# Monthly and total charges analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Monthly charges distribution by churn
for churn_val, label, color in [(0, 'Retained', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
    axes[0, 0].hist(df[df['Churn']==churn_val]['MonthlyCharges'], bins=30, alpha=0.6, 
                    label=label, color=color, edgecolor='black')
axes[0, 0].set_xlabel('Monthly Charges ($)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Monthly Charges Distribution by Churn', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Monthly charges boxplot
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[0, 1], patch_artist=True)
axes[0, 1].set_xlabel('Churn Status', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Monthly Charges ($)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Monthly Charges by Churn Status', fontsize=12, fontweight='bold')
axes[0, 1].set_xticklabels(['Retained', 'Churned'])
plt.suptitle('')

# Total charges distribution
for churn_val, label, color in [(0, 'Retained', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
    axes[1, 0].hist(df[df['Churn']==churn_val]['TotalCharges'], bins=30, alpha=0.6, 
                    label=label, color=color, edgecolor='black')
axes[1, 0].set_xlabel('Total Charges ($)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Total Charges Distribution by Churn', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Charge groups analysis
df['ChargeGroup'] = pd.cut(df['MonthlyCharges'], bins=[0, 40, 60, 80, 120], 
                            labels=['Low ($20-40)', 'Medium ($40-60)', 'High ($60-80)', 'Very High ($80+)'])
charge_churn = pd.crosstab(df['ChargeGroup'], df['Churn'], normalize='index') * 100
charge_churn.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[1, 1].set_xlabel('Monthly Charge Group', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Percentage', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Churn Rate by Charge Group', fontsize=12, fontweight='bold')
axes[1, 1].legend(['Retained', 'Churned'])
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45, ha='right')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/05_financial_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFinancial Statistics by Churn Status:")
print("=" * 60)
print(df.groupby('Churn')[['MonthlyCharges', 'TotalCharges']].describe())

## 8. Customer Engagement Metrics

In [ ]:
# Satisfaction and support analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Satisfaction score distribution
for churn_val, label, color in [(0, 'Retained', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
    axes[0].hist(df[df['Churn']==churn_val]['SatisfactionScore'], bins=30, alpha=0.6, 
                 label=label, color=color, edgecolor='black')
axes[0].set_xlabel('Satisfaction Score', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('Customer Satisfaction by Churn Status', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Support tickets distribution
for churn_val, label, color in [(0, 'Retained', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
    ticket_dist = df[df['Churn']==churn_val]['SupportTickets'].value_counts().sort_index()
    axes[1].plot(ticket_dist.index, ticket_dist.values, marker='o', label=label, 
                 color=color, linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Support Tickets', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Number of Customers', fontsize=11, fontweight='bold')
axes[1].set_title('Support Tickets by Churn Status', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Correlation: satisfaction vs support tickets
scatter = axes[2].scatter(df['SupportTickets'], df['SatisfactionScore'], 
                          c=df['Churn'], cmap='RdYlGn_r', alpha=0.5, s=20, edgecolors='black', linewidth=0.5)
axes[2].set_xlabel('Support Tickets', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Satisfaction Score', fontsize=11, fontweight='bold')
axes[2].set_title('Satisfaction vs Support Tickets', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)
cbar = plt.colorbar(scatter, ax=axes[2])
cbar.set_label('Churn', fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/06_engagement_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nEngagement Metrics by Churn:")
print("=" * 60)
print(df.groupby('Churn')[['SatisfactionScore', 'SupportTickets']].describe())

## 9. Correlation Analysis

In [ ]:
# Prepare numerical features for correlation
numerical_features = ['Age', 'SeniorCitizen', 'TenureMonths', 'PhoneService', 'MultipleLines',
                      'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                      'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges',
                      'TotalCharges', 'SatisfactionScore', 'SupportTickets', 'ServiceCount', 'Churn']

# Calculate correlation matrix
correlation_matrix = df[numerical_features].corr()

# Create heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../visualizations/07_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Top correlations with churn
churn_corr = correlation_matrix['Churn'].sort_values(ascending=False)
print("\nTop Features Correlated with Churn:")
print("=" * 60)
print(churn_corr[1:])

## 10. Key Insights and Summary

### Main Findings:

1. **Churn Rate**: Overall churn rate is approximately 25-30%, indicating significant customer retention challenges

2. **Contract Type Impact**: 
   - Month-to-month contracts have the highest churn rate (~45%)
   - Long-term contracts (1-2 years) show much lower churn (~10-15%)
   - **Recommendation**: Incentivize customers to commit to longer contracts

3. **Tenure Matters**:
   - New customers (0-12 months) are at highest risk
   - Churn decreases significantly after 24 months
   - **Recommendation**: Focus retention efforts on first year customers

4. **Service Add-ons Reduce Churn**:
   - Customers with online security, tech support, and backup services churn less
   - More services = lower churn rate
   - **Recommendation**: Promote bundled service packages

5. **Pricing Sensitivity**:
   - Higher monthly charges correlate with increased churn
   - Customers paying $70+ have higher churn risk
   - **Recommendation**: Review pricing strategy and offer competitive plans

6. **Payment Method**:
   - Electronic check users have highest churn
   - Automated payments (bank transfer) show lower churn
   - **Recommendation**: Encourage automatic payment enrollment

7. **Customer Satisfaction**:
   - Strong negative correlation between satisfaction and churn
   - Support tickets inversely related to satisfaction
   - **Recommendation**: Improve customer service and reduce support ticket incidents

### Next Steps:
- Build predictive models to identify at-risk customers
- Develop targeted retention campaigns
- Implement early warning system for new customers

In [ ]:
# Save cleaned data for modeling
df.to_csv('../processed/customer_churn_cleaned.csv', index=False)
print("✓ Cleaned data saved for modeling phase")